## ----------------------------------------------------
## RECREACIÒN DE SILVER 2021 DE MANERA CORRECTA
## ----------------------------------------------------

In [ ]:
# ============================================
# 🧱 CONFIG
# ============================================

from pyspark.sql.functions import (
    input_file_name, col, lower, split, size
)

bronze_path = "Files/raw/dane/year=2026"

# ============================================
# 📥 1. LEER RAW
# ============================================

df_raw = spark.read \
    .format("text") \
    .option("recursiveFileLookup", "true") \
    .load(bronze_path)

df_raw = df_raw.withColumn("file_name", input_file_name())

print("✅ Archivos leídos")

# ============================================
# 🧠 2. IDENTIFICAR DATASETS
# ============================================

df_raw = df_raw.withColumn(
    "file_name_clean",
    lower(col("file_name"))
)

# ============================================
# 🔍 3. CLASIFICACIÓN AMPLIA
# ============================================

from pyspark.sql.functions import when

df_raw = df_raw.withColumn(
    "dataset",
    when(col("file_name_clean").contains("ocupados"), "ocupados")
    .when(col("file_name_clean").contains("no%20ocupados"), "no_ocupados")
    .when(col("file_name_clean").contains("fuerza"), "fuerza_trabajo")
    .when(col("file_name_clean").contains("hogar"), "hogar")
    .otherwise("otros")
)

# ============================================
# 🔬 4. SPLIT COLUMNAS
# ============================================

df_raw = df_raw.withColumn(
    "cols",
    split(col("value"), ";")
)

df_raw = df_raw.withColumn(
    "num_columns",
    size(col("cols"))
)

# ============================================
# 📊 5. DISTRIBUCIÓN
# ============================================

print("📊 Distribución por dataset:")
df_raw.groupBy("dataset", "num_columns").count().show(100, False)

# ============================================
# 🧠 6. EXTRAER HEADER POR DATASET
# ============================================

datasets = ["ocupados", "no_ocupados", "fuerza_trabajo"]

for ds in datasets:
    print(f"\n🔎 DATASET: {ds}")
    
    df_ds = df_raw.filter(col("dataset") == ds)
    
    header = df_ds \
        .filter(col("value").rlike("MES|AREA|FT|OCI")) \
        .select("cols") \
        .first()
    
    if header:
        cols = header[0]
        print(f"✔ Columnas detectadas: {len(cols)}")
        print(cols[:20])  # primeras columnas
    else:
        print("❌ No se detectó header")

# ============================================
# 🔍 7. SAMPLE REAL POR DATASET
# ============================================

for ds in datasets:
    print(f"\n📄 SAMPLE: {ds}")
    
    df_raw.filter(col("dataset") == ds) \
        .select("value") \
        .show(5, False)

StatementMeta(, bfe61bf0-c0e7-4255-89ce-1a643ac0cbd9, 31, Finished, Available, Finished, False)

✅ Archivos leídos
📊 Distribución por dataset:


+--------------+-----------+------+
|dataset       |num_columns|count |
+--------------+-----------+------+
|ocupados      |208        |58253 |
|otros         |112        |107620|
|otros         |55         |134344|
|otros         |43         |134344|
|otros         |60         |107620|
|fuerza_trabajo|42         |107620|
|hogar         |49         |48607 |
|ocupados      |36         |49369 |
+--------------+-----------+------+


🔎 DATASET: ocupados
✔ Columnas detectadas: 208
['PERIODO', 'MES', 'PER', 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'HOGAR', 'REGIS', 'AREA', 'CLASE', 'FEX_C18', 'DPTO', 'FT', 'P3044S2', 'P6440', 'P6450', 'P6460', 'P6460S1', 'P6400', 'P6410']

🔎 DATASET: no_ocupados


❌ No se detectó header

🔎 DATASET: fuerza_trabajo


✔ Columnas detectadas: 42
['PERIODO', 'MES', 'PER', 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'HOGAR', 'REGIS', 'AREA', 'CLASE', 'FEX_C18', 'DPTO', 'FT', 'FFT', 'PET', 'P6240', 'P6240S1', 'P6240S2', 'P6250', 'P6260']

📄 SAMPLE: ocupados
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

+-----+
|value|
+-----+
+-----+


📄 SAMPLE: fuerza_trabajo


+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                                                                                                                          |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|PERIODO;MES;PER;DIRECTORIO;SECUENCIA_P;ORDEN;HOGAR;REGIS;AREA;CLASE;FEX_C18;DPTO;FT;FFT;PET;P6240;P6240S1;P6240S2;P6250;P6260;P62

In [5]:
# ============================================
# 🧱 CONFIG
# ============================================

from pyspark.sql.functions import (
    col, when, input_file_name, lower, trim,
    lit, regexp_extract, current_timestamp,
    to_json, struct
)
from pyspark.sql.types import StringType

bronze_path = "Files/raw/dane"

silver_table = "dane_silver_lh.labor_silver_real"
quarantine_table = "dane_silver_lh.labor_quarantine"

# ============================================
# 📥 1. READ BRONZE
# ============================================

df = spark.read \
    .format("csv") \
    .option("header", True) \
    .option("delimiter", ";") \
    .option("recursiveFileLookup", "true") \
    .load(bronze_path)

print("✅ Archivos leídos")

# ============================================
# 🧠 2. FILE NAME NORMALIZADO
# ============================================

df = df.withColumn("file_name", lower(input_file_name()))

# ============================================
# 🧠 3. STATUS DESDE ARCHIVO
# ============================================

df = df.withColumn(
    "status_from_file",
    when(col("file_name").rlike("no.?ocupados|desocupados"), "desocupado")
    .when(col("file_name").rlike("ocupados"), "ocupado")
)

# ============================================
# 🧠 4. DETECTAR SI EXISTE DSI
# ============================================

has_dsi = "DSI" in df.columns

if has_dsi:
    df = df.withColumn("DSI_clean", trim(col("DSI")))

    df = df.withColumn(
        "status_from_dsi",
        when(col("DSI_clean") == "1", "ocupado")
        .when(col("DSI_clean") == "2", "desocupado")
        .when(col("DSI_clean") == "3", "inactivo")
    )
else:
    df = df.withColumn("status_from_dsi", lit(None).cast(StringType()))

# ============================================
# 🧠 5. STATUS FINAL UNIFICADO
# ============================================

df = df.withColumn(
    "status",
    when(col("status_from_file").isNotNull(), col("status_from_file"))
    .otherwise(col("status_from_dsi"))
)

print("✅ Status unificado")

# ============================================
# 🧠 6. EXTRAER YEAR DESDE PATH
# ============================================

df = df.withColumn(
    "year",
    regexp_extract(col("file_name"), r"year=(\d{4})", 1).cast("int")
)

# ============================================
# 🧠 7. SELECCIÓN SEGURA DE COLUMNAS
# ============================================

# Validar columnas mínimas
required_cols = ["MES", "AREA", "DPTO"]

for c in required_cols:
    if c not in df.columns:
        raise Exception(f"❌ Columna {c} no existe en dataset")

# Detectar si existe peso (encuestas)
has_weight = "fex_c_2011" in df.columns

# ============================================
# 🧱 8. NORMALIZAR
# ============================================

df_clean = df.select(
    col("MES").cast("int").alias("month"),
    col("AREA").alias("area_code"),
    col("DPTO").alias("depto_code"),
    col("status"),
    col("year"),
    col("file_name"),
    col("fex_c_2011").cast("double").alias("weight") if has_weight else lit(1.0).alias("weight")
)

# ============================================
# 🧪 9. DATA QUALITY
# ============================================

df_valid = df_clean.filter(
    col("month").between(1, 12) &
    col("year").isNotNull() &
    col("status").isNotNull()
)

df_invalid = df_clean.subtract(df_valid)

valid_count = df_valid.count()
invalid_count = df_invalid.count()

print(f"✔ Registros válidos: {valid_count}")
print(f"❌ Registros inválidos: {invalid_count}")

# ============================================
# 🚨 10. QUARANTINE
# ============================================

if invalid_count > 0:
    df_quarantine = df_invalid.select(
        to_json(struct(*df_invalid.columns)).alias("raw_payload"),
        current_timestamp().alias("detected_at")
    )

    df_quarantine.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(quarantine_table)

    print(f"⚠️ Enviados a quarantine: {invalid_count}")
else:
    print("✅ No hay inválidos")

# ============================================
# 🧮 11. AGREGACIÓN CORRECTA
# ============================================

df_agg = df_valid.groupBy(
    "year",
    "month",
    "area_code",
    "depto_code",
    "status"
).sum("weight").withColumnRenamed("sum(weight)", "population")

print("✅ Agregación lista")

# ============================================
# 💾 12. WRITE SILVER
# ============================================

df_agg.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_table)

print("✅ Silver creado correctamente")

# ============================================
# 🧪 13. VALIDACIÓN FINAL
# ============================================

spark.table(silver_table).show(50)

StatementMeta(, 933615df-2f78-4167-89d1-af181fca4610, 7, Finished, Available, Finished, False)

✅ Archivos leídos
✅ Status unificado
✔ Registros válidos: 760164
❌ Registros inválidos: 215716
⚠️ Enviados a quarantine: 215716
✅ Agregación lista
✅ Silver creado correctamente
+----+-----+---------+--------------------+-------+----------+
|year|month|area_code|          depto_code| status|population|
+----+-----+---------+--------------------+-------+----------+
|2022|    9|        2|                NULL|ocupado|    1148.0|
|2022|    7|        1|                NULL|ocupado|    1878.0|
|2022|   10|        2|                NULL|ocupado|    1240.0|
|2022|   12|        2|                NULL|ocupado|   15617.0|
|2022|    8|        2|                NULL|ocupado|    2518.0|
|2022|    8|        1|                NULL|ocupado|    4420.0|
|2022|   12|        1|                NULL|ocupado|   26430.0|
|2022|   11|        1|                NULL|ocupado|     963.0|
|2022|   11|        2|                NULL|ocupado|     488.0|
|2022|    9|        1|                NULL|ocupado|    1986.0|
|202

## EXTENDER COLUMN MAPPING

In [ ]:
spark.sql("""
ALTER TABLE column_mapping
ADD COLUMNS (
    is_critical BOOLEAN,
    allow_type_change BOOLEAN
)
""")

StatementMeta(, d5853226-fdd6-4ebb-84f2-a0fbd376d9c7, -1, Cancelled, , Cancelled, True)

## CONFIGURAR CONTRATO

In [ ]:
spark.sql("""
UPDATE column_mapping
SET 
    is_critical = true,
    allow_type_change = false
WHERE source_column IN ('Year', 'Month')
""")

spark.sql("""
UPDATE column_mapping
SET 
    is_critical = false,
    allow_type_change = true
WHERE source_column IN ('Metric', 'Geography')
""")

StatementMeta(, d5853226-fdd6-4ebb-84f2-a0fbd376d9c7, -1, Cancelled, , Cancelled, True)

## CREAR TABLA DE CUARENTENA

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS dane_silver_lh.labor_quarantine (
    raw_payload STRING,
    error_reason STRING,
    detected_at TIMESTAMP
)
USING DELTA
""")

StatementMeta(, d5853226-fdd6-4ebb-84f2-a0fbd376d9c7, -1, Cancelled, , Cancelled, True)

## DEFINIR REGLAS DE CALIDAD

In [ ]:
## Year → no null + numérico
## Month → no null + entre 1 y 12

StatementMeta(, d5853226-fdd6-4ebb-84f2-a0fbd376d9c7, -1, Cancelled, , Cancelled, True)

In [16]:
# ============================================
# 🧱 CONFIG
# ============================================

from pyspark.sql.functions import col, input_file_name, lower, when, regexp_extract

bronze_path = "Files/raw/dane"
silver_table = "dane_silver_lh.labor_silver_clean"

# ============================================
# 📥 1. LEER CSV
# ============================================

df = spark.read \
    .format("csv") \
    .option("header", True) \
    .option("delimiter", ";") \
    .option("recursiveFileLookup", "true") \
    .load(bronze_path)

df = df.withColumn("file_name", input_file_name())

print("✅ Archivos leídos")

# ============================================
# 🔥 2. FILTRAR SOLO DATA RELEVANTE
# ============================================

df = df.filter(
    (lower(col("file_name")).rlike("ocupados|desocupados|no.?ocupados")) &
    (~lower(col("file_name")).rlike("directorio|secuencia|hogar"))
)

print("✅ Archivos filtrados")

# ============================================
# 🧠 3. NORMALIZAR
# ============================================

df_clean = df.select(
    col("MES").cast("int").alias("month"),
    col("AREA").alias("area_code"),
    col("DPTO").alias("depto_code"),
    col("file_name")
)

# ============================================
# 🧠 4. STATUS
# ============================================

df_clean = df_clean.withColumn(
    "status",
    when(lower(col("file_name")).rlike("no.?ocupados|desocupados"), "desocupado")
    .when(lower(col("file_name")).rlike("ocupados"), "ocupado")
)

# ============================================
# 🧠 5. YEAR DESDE PATH
# ============================================

df_clean = df_clean.withColumn(
    "year",
    regexp_extract(col("file_name"), r"year=(\d{4})", 1).cast("int")
)

print("✅ Silver limpio listo")

# ============================================
# 💾 6. WRITE SILVER
# ============================================

df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(silver_table)

print("✅ Silver guardado")

spark.table(silver_table).show(10)

StatementMeta(, d5853226-fdd6-4ebb-84f2-a0fbd376d9c7, 18, Finished, Available, Finished, False)

✅ Archivos leídos
✅ Archivos filtrados
✅ Silver limpio listo
✅ Silver guardado
+-----+---------+----------+--------------------+-------+----+
|month|area_code|depto_code|           file_name| status|year|
+-----+---------+----------+--------------------+-------+----+
| NULL|        1|      NULL|abfss://f5879c16-...|ocupado|2024|
| NULL|        3|      NULL|abfss://f5879c16-...|ocupado|2024|
| NULL|        1|      NULL|abfss://f5879c16-...|ocupado|2024|
| NULL|        2|      NULL|abfss://f5879c16-...|ocupado|2024|
| NULL|        1|      NULL|abfss://f5879c16-...|ocupado|2024|
| NULL|        3|      NULL|abfss://f5879c16-...|ocupado|2024|
| NULL|        1|      NULL|abfss://f5879c16-...|ocupado|2024|
| NULL|        2|      NULL|abfss://f5879c16-...|ocupado|2024|
| NULL|        2|      NULL|abfss://f5879c16-...|ocupado|2024|
| NULL|        2|      NULL|abfss://f5879c16-...|ocupado|2024|
+-----+---------+----------+--------------------+-------+----+
only showing top 10 rows



In [5]:
from pyspark.sql.functions import input_file_name

df_files = spark.read \
    .format("binaryFile") \
    .option("recursiveFileLookup", "true") \
    .load("Files/raw/dane")

df_files.select("path").show(50, False)

StatementMeta(, d5853226-fdd6-4ebb-84f2-a0fbd376d9c7, 7, Finished, Available, Finished, False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                                                                                                                                                                                                     |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|abfss://f5879c16-6832-4003-90dd-eb2c10497cc0@onelake.dfs.fabric.microsoft.com/ecf22981-1012-4b17-861f-62880aa61238/Files/raw/dane/year=2021/month=07/Cabecera - Ocupados.csv                                             |
|abfss://f5879c16-6832-4003-90dd-eb2c10497cc0@onelake.dfs.fabric.microsoft.com/ecf22981-1012-4b17-861f-62880aa61238/File

In [6]:
from pyspark.sql.functions import lower

df_files = df_files.withColumn("path_lower", lower(col("path")))

df_files = df_files.withColumn(
    "file_type",
    when(col("path_lower").rlike("ocupados"), "ocupados")
    .when(col("path_lower").rlike("desocupados"), "desocupados")
    .when(col("path_lower").rlike("no.?ocupados"), "desocupados")
    .otherwise("microdata")
)

df_files.groupBy("file_type").count().show()

StatementMeta(, d5853226-fdd6-4ebb-84f2-a0fbd376d9c7, 8, Finished, Available, Finished, False)

+---------+-----+
|file_type|count|
+---------+-----+
|microdata| 1019|
| ocupados|  334|
+---------+-----+



In [7]:
df_paths = df_files.filter(col("file_type") != "microdata")

paths = [row["path"] for row in df_paths.select("path").collect()]

StatementMeta(, d5853226-fdd6-4ebb-84f2-a0fbd376d9c7, 9, Finished, Available, Finished, False)

In [8]:
df = spark.read \
    .format("csv") \
    .option("header", True) \
    .option("delimiter", ";") \
    .load(paths)

StatementMeta(, d5853226-fdd6-4ebb-84f2-a0fbd376d9c7, 10, Finished, Available, Finished, False)

In [9]:
from pyspark.sql.functions import col, lower, regexp_extract

# ============================================
# 📂 LEER TODOS LOS ARCHIVOS DEL DATALAKE
# ============================================

df_files = spark.read \
    .format("binaryFile") \
    .option("recursiveFileLookup", "true") \
    .load("Files/")

print("✅ Inventario completo generado")

df_files.select("path").show(20, False)

StatementMeta(, d5853226-fdd6-4ebb-84f2-a0fbd376d9c7, 11, Finished, Available, Finished, False)

✅ Inventario completo generado
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                                                                                                                                                                                                     |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|abfss://f5879c16-6832-4003-90dd-eb2c10497cc0@onelake.dfs.fabric.microsoft.com/ecf22981-1012-4b17-861f-62880aa61238/Files/raw/dane_projections/year=2020/Fex proyeccion CNPV_2018_2020.dta                                |
|abfss://f5879c16-6832-4003-90dd-eb2c10497cc0@onelake.dfs.fabric.microsoft.com/ecf22981-1

In [10]:
# ============================================
# 🧠 EXTRAER METADATA DEL PATH
# ============================================

df_files = df_files.withColumn("path_lower", lower(col("path")))

df_files = df_files.withColumn(
    "year",
    regexp_extract(col("path"), r"year=(\d{4})", 1).cast("int")
)

df_files = df_files.withColumn(
    "month",
    regexp_extract(col("path"), r"month=(\d{2})", 1).cast("int")
)

print("✅ Metadata extraída")

df_files.select("path", "year", "month").show(20, False)

StatementMeta(, d5853226-fdd6-4ebb-84f2-a0fbd376d9c7, 12, Finished, Available, Finished, False)

✅ Metadata extraída
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+-----+
|path                                                                                                                                                                                                                     |year|month|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+-----+
|abfss://f5879c16-6832-4003-90dd-eb2c10497cc0@onelake.dfs.fabric.microsoft.com/ecf22981-1012-4b17-861f-62880aa61238/Files/raw/dane_projections/year=2020/Fex proyeccion CNPV_2018_2020.dta                                |2020|NULL |
|abfss://f5879c16-6832-4003-90dd-eb2c10497cc0@onelake.df

In [11]:
from pyspark.sql.functions import when

# ============================================
# 🧠 CLASIFICACIÓN ROBUSTA
# ============================================

df_files = df_files.withColumn(
    "file_type",
    
    # 🔴 desocupados (variantes)
    when(col("path_lower").rlike("des.?ocup|no.?ocup|desemple"), "desocupados")
    
    # 🟢 ocupados
    .when(
        (col("path_lower").rlike("ocup")) &
        (~col("path_lower").rlike("des.?ocup|no.?ocup")),
        "ocupados"
    )
    
    # 🔵 microdata (encuestas)
    .when(col("path_lower").rlike("directorio|secuencia|hogar|p[0-9]"), "microdata")
    
    # ⚫ otros
    .otherwise("otros")
)

print("✅ Clasificación completa")

df_files.groupBy("file_type").count().show()

StatementMeta(, d5853226-fdd6-4ebb-84f2-a0fbd376d9c7, 13, Finished, Available, Finished, False)

✅ Clasificación completa
+-----------+-----+
|  file_type|count|
+-----------+-----+
|      otros|  856|
|   ocupados|  170|
|  microdata|  165|
|desocupados|  165|
+-----------+-----+



In [12]:
df_files = df_files.withColumn(
    "file_type",
    
    # 🔴 desocupados (TODAS las variantes)
    when(col("path_lower").rlike("no.?ocup|des.?ocup"), "desocupados")
    
    # 🟢 ocupados
    .when(
        (col("path_lower").rlike("ocup")) &
        (~col("path_lower").rlike("no.?ocup|des.?ocup")),
        "ocupados"
    )
    
    # 🔵 microdata
    .when(col("path_lower").rlike("directorio|secuencia|hogar"), "microdata")
    
    # ⚫ otros datasets (ignorar)
    .otherwise("otros")
)

StatementMeta(, d5853226-fdd6-4ebb-84f2-a0fbd376d9c7, 14, Finished, Available, Finished, False)

In [13]:
df_target = df_files.filter(
    col("file_type").isin("ocupados", "desocupados")
)

StatementMeta(, d5853226-fdd6-4ebb-84f2-a0fbd376d9c7, 15, Finished, Available, Finished, False)

In [14]:
df_target.groupBy("year", "month", "file_type") \
    .count() \
    .orderBy("year", "month") \
    .show(200, False)

StatementMeta(, d5853226-fdd6-4ebb-84f2-a0fbd376d9c7, 16, Finished, Available, Finished, False)

+----+-----+-----------+-----+
|year|month|file_type  |count|
+----+-----+-----------+-----+
|2018|1    |ocupados   |3    |
|2018|1    |desocupados|3    |
|2018|2    |ocupados   |3    |
|2018|2    |desocupados|3    |
|2018|3    |ocupados   |3    |
|2018|3    |desocupados|3    |
|2018|4    |ocupados   |3    |
|2018|4    |desocupados|3    |
|2018|5    |ocupados   |3    |
|2018|5    |desocupados|3    |
|2018|6    |ocupados   |3    |
|2018|6    |desocupados|3    |
|2018|7    |ocupados   |3    |
|2018|7    |desocupados|3    |
|2018|8    |ocupados   |3    |
|2018|8    |desocupados|3    |
|2018|9    |ocupados   |3    |
|2018|9    |desocupados|3    |
|2018|10   |ocupados   |3    |
|2018|10   |desocupados|3    |
|2018|11   |ocupados   |3    |
|2018|11   |desocupados|3    |
|2018|12   |ocupados   |3    |
|2018|12   |desocupados|3    |
|2019|1    |ocupados   |3    |
|2019|1    |desocupados|3    |
|2019|2    |ocupados   |3    |
|2019|2    |desocupados|3    |
|2019|3    |ocupados   |3    |
|2019|3 

## DEFINIR COLUMNAS VALIOSAS

In [15]:
from pyspark.sql.functions import when

df_clean = df.select(
    col("MES").cast("int").alias("month"),
    col("AREA").alias("area_code"),
    col("DPTO").alias("depto_code"),
    col("file_name")
)

StatementMeta(, d5853226-fdd6-4ebb-84f2-a0fbd376d9c7, 17, Finished, Available, Finished, False)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `file_name` cannot be resolved. Did you mean one of the following? [`AREA`, `CLASE`, `DPTO`, `HOGAR`, `INGLABO`].;
'Project [cast(MES#3135 as int) AS month#3440, AREA#2981 AS area_code#3441, DPTO#3139 AS depto_code#3442, 'file_name]
+- Relation [DIRECTORIO#2976,SECUENCIA_P#2977,ORDEN#2978,HOGAR#2979,REGIS#2980,AREA#2981,CLASE#2982,P388#2983,P6440#2984,P6450#2985,P6460#2986,P6460S1#2987,P6400#2988,P6410#2989,P6410S1#2990,P6422#2991,P6424S1#2992,P6424S2#2993,P6424S3#2994,P6426#2995,P6430S1#2996,P6480#2997,P6480S1#2998,P9440#2999,... 142 more fields] csv


## DERIVAR STATUS

In [ ]:
from pyspark.sql.functions import lower

df_clean = df_clean.withColumn(
    "status",
    when(lower(col("file_name")).rlike("no.?ocupados|desocupados"), "desocupado")
    .when(lower(col("file_name")).rlike("ocupados"), "ocupado")

StatementMeta(, d5853226-fdd6-4ebb-84f2-a0fbd376d9c7, -1, Cancelled, , Cancelled, True)

## RECONSTRUCCIÓN TOTAL SILVER 2019

In [ ]:
## ============================================================================
## 🚀 SILVER 2019: RECONSTRUCCIÓN TOTAL (ANTI-CAOS)
## ============================================================================
from pyspark.sql import functions as F
from pyspark.sql.types import *

year = 2019
bronze_path = f"Files/raw/dane/year={year}/" # Asegúrate de que termine en /
silver_table = f"dane_silver_lh.labor_{year}"

print(f"🕵️‍♂️ Iniciando escaneo de {year}...")

# 1. Leemos TODO como texto plano para evitar errores de PATH_NOT_FOUND individuales
df_raw = spark.read.format("text").option("recursiveFileLookup", "true").load(bronze_path) \
              .withColumn("file_name", F.input_file_name()) \
              .withColumn("fn_low", F.lower(F.col("file_name")))

# 2. Clasificación por contenido (No por nombre exacto)
# Buscamos patrones: si tiene "deso", es desocupado. Si tiene "ocu" pero no "deso", es ocupado.
df_meta = df_raw.withColumn(
    "status_file",
    F.when(F.col("fn_low").rlike("deso|no%20ocu|no_ocu"), "desocupado")
    .when(F.col("fn_low").rlike("ocupa"), "ocupado")
    .otherwise("otro")
).filter(F.col("status_file").isin("ocupado", "desocupado")) \
 .filter(~F.col("fn_low").rlike("caracteristicas|ingresos|vivienda|fuerza|seguridad|formas"))

# 3. Función Mágica para procesar el desorden
def process_2019_chaos(df_input, label):
    df_group = df_input.filter(F.col("status_file") == label)
    if df_group.count() == 0: 
        print(f"⚠️ No encontré datos para {label}")
        return None
    
    # Extraemos la primera fila para detectar separador y columnas
    first_row = df_group.filter(F.upper(F.col("value")).contains("DIRECTORIO")).limit(1).collect()
    if not first_row: return None
    
    header_val = first_row[0]['value']
    delim = ";" if ";" in header_val else ","
    cols = [c.strip().upper() for c in header_val.split(delim)]
    
    # Buscamos el Peso (FEX) dinámicamente
    try:
        # En 2019 suele ser FEX_C_2011 o FEX_C
        idx_peso = next(i for i, c in enumerate(cols) if "FEX" in c or "PESO" in c)
        print(f"   ✅ {label}: Separador '{delim}' | Columna peso: '{cols[idx_peso]}' (idx {idx_peso})")
    except StopIteration:
        print(f"   ❌ {label}: No encontré columna de peso en: {cols[:5]}...")
        return None

    # Procesamos
    return df_group.filter(~F.upper(F.col("value")).contains("DIRECTORIO")) \
        .withColumn("month", F.regexp_extract(F.col("file_name"), r"month=(\d+)", 1).cast("int")) \
        .withColumn("split_data", F.split(F.col("value"), delim)) \
        .withColumn("geo_source", F.when(F.col("fn_low").rlike("cabe|urban"), "cabecera")
                                 .when(F.col("fn_low").rlike("resto|rural"), "resto")
                                     .otherwise("area")) \
        .select(
            F.lit(year).alias("year"),
            F.col("month"),
            F.lit(label).alias("status"),
            F.col("geo_source"),
            # Limpiamos el peso de comillas y basura
            F.regexp_replace(F.regexp_replace(F.col("split_data")[idx_peso], '"', ''), ",", ".").cast("double").alias("weight")
        ).filter(F.col("weight").isNotNull())

# 4. Ejecución Quirúrgica
df_ocu = process_2019_chaos(df_meta, "ocupado")
df_des = process_2019_chaos(df_meta, "desocupado")

# 5. Unión y Guardado
final_dfs = [df for df in [df_ocu, df_des] if df is not None]
if final_dfs:
    df_silver_2019 = final_dfs[0]
    if len(final_dfs) > 1:
        df_silver_2019 = df_silver_2019.unionByName(final_dfs[1])
    
    # Flag para Gold
    df_silver_2019 = df_silver_2019.withColumn("is_national_total", F.col("geo_source").isin("cabecera", "resto"))
    
    spark.sql(f"DROP TABLE IF EXISTS {silver_table}")
    df_silver_2019.write.format("delta").mode("overwrite").saveAsTable(silver_table)
    
    print(f"\n📊 RESUMEN SILVER {year}:")
    df_silver_2019.groupBy("status", "month").count().orderBy("month", "status").show()
else:
    print("❌ Error fatal: No se pudo procesar nada de 2019.")

StatementMeta(, ca0ff25c-c42f-481a-8f49-50adb8c09a33, 3, Finished, Available, Finished, False)

🕵️‍♂️ Iniciando escaneo de 2019...


   ✅ ocupado: Separador ';' | Columna peso: 'FEX_C_2011' (idx 164)


   ✅ desocupado: Separador ';' | Columna peso: 'FEX_C_2011' (idx 41)



📊 RESUMEN SILVER 2019:


+----------+-----+-----+
|    status|month|count|
+----------+-----+-----+
|desocupado|    1| 6401|
|   ocupado|    1|40864|
|desocupado|    2| 6024|
|   ocupado|    2|42307|
|desocupado|    3| 5738|
|   ocupado|    3|41908|
|desocupado|    4| 5572|
|   ocupado|    4|41995|
|desocupado|    5| 5703|
|   ocupado|    5|42990|
|desocupado|    6| 5280|
|   ocupado|    6|42076|
|desocupado|    7| 5406|
|   ocupado|    7|42409|
|desocupado|    8| 5721|
|   ocupado|    8|42716|
|desocupado|    9| 5220|
|   ocupado|    9|41959|
|desocupado|   10| 5263|
|   ocupado|   10|42871|
+----------+-----+-----+
only showing top 20 rows



## SILVER 2020- RECONSTRUIDO

In [ ]:
## ============================================================================
## 🚀 SILVER 2020: LA BATALLA CONTRA EL "ASCO" DE LA PANDEMIA
## ============================================================================
from pyspark.sql import functions as F

year = 2020
bronze_path = f"Files/raw/dane/year={year}/"
silver_table = f"dane_silver_lh.labor_{year}"

print(f"🕵️‍♂️ Iniciando escaneo profundo de {year}...")

# 1. Lectura masiva (Anti-PathNotFound)
df_raw = spark.read.format("text").option("recursiveFileLookup", "true").load(bronze_path) \
              .withColumn("file_name", F.input_file_name()) \
              .withColumn("fn_low", F.lower(F.col("file_name")))

# 2. Clasificación por contenido
# En 2020 aparecen archivos de "Fuerza de Trabajo" que hay que ignorar
df_meta = df_raw.withColumn(
    "status_file",
    F.when(F.col("fn_low").rlike("deso|no%20ocu|no_ocu"), "desocupado")
    .when(F.col("fn_low").rlike("ocupa"), "ocupado")
    .otherwise("otro")
).filter(F.col("status_file").isin("ocupado", "desocupado")) \
 .filter(~F.col("fn_low").rlike("caracteristicas|ingresos|vivienda|fuerza|seguridad|formas|inactivos"))

# 3. Procesador de Caos 2020
def process_2020_chaos(df_input, label):
    df_group = df_input.filter(F.col("status_file") == label)
    if df_group.count() == 0: 
        print(f"⚠️ No hay datos para {label}")
        return None
    
    # Buscamos el header (DIRECTORIO)
    header_row = df_group.filter(F.upper(F.col("value")).contains("DIRECTORIO")).limit(1).collect()
    if not header_row: return None
    
    val = header_row[0]['value']
    delim = ";" if ";" in val else ","
    cols = [c.strip().upper() for c in val.split(delim)]
    
    try:
        # En 2020 el nombre suele ser 'FEX_C' o 'FEX_C18'
        idx_peso = next(i for i, c in enumerate(cols) if "FEX" in c or "PESO" in c)
        print(f"   ✅ {label}: Delim '{delim}' | Col '{cols[idx_peso]}' (idx {idx_peso})")
    except StopIteration:
        print(f"   ❌ {label}: Sin columna de peso en {cols[:5]}")
        return None

    return df_group.filter(~F.upper(F.col("value")).contains("DIRECTORIO")) \
        .withColumn("month", F.regexp_extract(F.col("file_name"), r"month=(\d+)", 1).cast("int")) \
        .withColumn("split_data", F.split(F.col("value"), delim)) \
        .withColumn("geo_source", F.when(F.col("fn_low").rlike("cabe|urban"), "cabecera")
                                 .when(F.col("fn_low").rlike("resto|rural"), "resto")
                                 .otherwise("area")) \
        .select(
            F.lit(year).alias("year"),
            F.col("month"),
            F.lit(label).alias("status"),
            F.col("geo_source"),
            F.regexp_replace(F.regexp_replace(F.col("split_data")[idx_peso], '"', ''), ",", ".").cast("double").alias("weight")
        ).filter(F.col("weight").isNotNull())

# 4. Ejecución
df_ocu = process_2020_chaos(df_meta, "ocupado")
df_des = process_2020_chaos(df_meta, "desocupado")

# 5. Unión y Guardado
final_dfs = [df for df in [df_ocu, df_des] if df is not None]
if final_dfs:
    df_silver_2020 = final_dfs[0]
    if len(final_dfs) > 1: df_silver_2020 = df_silver_2020.unionByName(final_dfs[1])
    
    df_silver_2020 = df_silver_2020.withColumn("is_national_total", F.col("geo_source").isin("cabecera", "resto"))
    
    spark.sql(f"DROP TABLE IF EXISTS {silver_table}")
    df_silver_2020.write.format("delta").mode("overwrite").saveAsTable(silver_table)
    
    print(f"\n📊 RESUMEN SILVER {year}:")
    df_silver_2020.groupBy("status", "month").count().orderBy("month", "status").show()
else:
    print("❌ Error: 2020 no se pudo procesar.")

StatementMeta(, ca0ff25c-c42f-481a-8f49-50adb8c09a33, 4, Finished, Available, Finished, False)

🕵️‍♂️ Iniciando escaneo profundo de 2020...


   ✅ ocupado: Delim ',' | Col 'FEX_C' (idx 9)


   ✅ desocupado: Delim ',' | Col 'FEX_C' (idx 9)



📊 RESUMEN SILVER 2020:


+----------+-----+-----+
|    status|month|count|
+----------+-----+-----+
|desocupado|    1| 4913|
|   ocupado|    1|28104|
|desocupado|    2| 4578|
|   ocupado|    2|28667|
|   ocupado|    3|21881|
|   ocupado|    4|21605|
|   ocupado|    5|22698|
|desocupado|    6| 7420|
|   ocupado|    6|23907|
|desocupado|    7| 7110|
|   ocupado|    7|23760|
|desocupado|    8| 6234|
|   ocupado|    8|25408|
|desocupado|    9| 5744|
|   ocupado|    9|25883|
|desocupado|   10| 5487|
|   ocupado|   10|27200|
|desocupado|   11| 4915|
|   ocupado|   11|26585|
|desocupado|   12| 4858|
+----------+-----+-----+
only showing top 20 rows



In [7]:
# 📂 LISTADO REAL DE CARPETAS DE MESES
year = 2021
path_root = f"Files/raw/dane/year={year}/"

print(f"📁 Analizando estructura de carpetas para {year}...")

# Listamos solo los nombres de los directorios
carpetas = mssparkutils.fs.ls(path_root)
for c in carpetas:
    if c.isDir:
        print(f"✅ Carpeta encontrada: '{c.name}'")

StatementMeta(, c54981d5-1cbc-4547-8959-0c17da3b1fb0, 9, Finished, Available, Finished, False)

📁 Analizando estructura de carpetas para 2021...
✅ Carpeta encontrada: 'month=01'
✅ Carpeta encontrada: 'month=02'
✅ Carpeta encontrada: 'month=03'
✅ Carpeta encontrada: 'month=04'
✅ Carpeta encontrada: 'month=05'
✅ Carpeta encontrada: 'month=06'
✅ Carpeta encontrada: 'month=07'
✅ Carpeta encontrada: 'month=08'
✅ Carpeta encontrada: 'month=09'
✅ Carpeta encontrada: 'month=10'
✅ Carpeta encontrada: 'month=11'
✅ Carpeta encontrada: 'month=12'


## SILVER 2021- RECONSTRUIDO

In [8]:
## ============================================================================
## 🚀 SILVER 2021: VERSIÓN DEFINITIVA (CASE-INSENSITIVE + ZERO-PADDING)
## ============================================================================
from pyspark.sql import functions as F

year = 2021
bronze_path = f"Files/raw/dane/year={year}/"
silver_table = f"dane_silver_lh.labor_{year}"

# 1. Buscamos TODO (usamos * sin extensión para no perder los .csv minúsculas)
all_files = spark.createDataFrame(
    spark.sparkContext.wholeTextFiles(f"{bronze_path}*/*", minPartitions=1)
).select(F.col("_1").alias("path")).filter(F.lower(F.col("path")).rlike("ocupa|deso")) \
 .filter(F.lower(F.col("path")).rlike("\.csv$")) \
 .filter(~F.lower(F.col("path")).rlike("caracteristicas|ingresos|vivienda|fuerza|seguridad|formas|inactivos"))

files_list = [row['path'] for row in all_files.collect()]
print(f"📂 Se encontraron {len(files_list)} archivos. Procesando la serie completa...")

final_list = []

for path in files_list:
    fn = path.lower()
    label = "desocupado" if any(x in fn for x in ["deso", "no%20ocu", "no_ocu"]) else "ocupado"
    geo = "cabecera" if any(x in fn for x in ["cabe", "urban"]) else ("resto" if any(x in fn for x in ["resto", "rural"]) else "area")
    
    # Extraemos el mes manejando el cero (ej: month=05 -> 5)
    month_str = path.split("month=")[1].split("/")[0]
    month = int(month_str)
    
    df_temp = spark.read.text(path)
    header_row = df_temp.filter(F.upper(F.col("value")).rlike("DIRECTORIO|SECUENCIA")).limit(1).collect()
    
    if not header_row: continue
    
    val = header_row[0]['value']
    delim = ";" if ";" in val else ","
    cols = [c.strip().upper() for c in val.split(delim)]
    
    try:
        idx_peso = next(i for i, c in enumerate(cols) if any(p in c for p in ["FEX", "PESO", "FACTOR"]))
        
        processed = df_temp.filter(~F.upper(F.col("value")).rlike("DIRECTORIO|SECUENCIA")) \
            .withColumn("split_data", F.split(F.col("value"), delim)) \
            .select(
                F.lit(year).alias("year"),
                F.lit(month).alias("month"),
                F.lit(label).alias("status"),
                F.lit(geo).alias("geo_source"),
                F.regexp_replace(F.regexp_replace(F.col("split_data")[idx_peso], '[ "]', ''), ",", ".").cast("double").alias("weight")
            ).filter(F.col("weight").isNotNull())
        
        final_list.append(processed)
    except:
        continue

# 2. Unión y Guardado
if final_list:
    df_silver_2021 = final_list[0]
    for d in final_list[1:]: df_silver_2021 = df_silver_2021.unionByName(d)
    
    df_silver_2021 = df_silver_2021.withColumn("is_national_total", F.col("geo_source").isin("cabecera", "resto"))
    
    spark.sql(f"DROP TABLE IF EXISTS {silver_table}")
    df_silver_2021.write.format("delta").mode("overwrite").saveAsTable(silver_table)
    
    print(f"\n📊 RESUMEN 2021 - ¡TODOS LOS MESES CAPTURADOS!")
    df_silver_2021.groupBy("month").agg(
        F.sum(F.when(F.col("status")=="ocupado", F.col("weight"))).alias("Ocupados_Brutos")
    ).orderBy("month").show()

StatementMeta(, c54981d5-1cbc-4547-8959-0c17da3b1fb0, 10, Finished, Available, Finished, False)

📂 Se encontraron 72 archivos. Procesando la serie completa...

📊 RESUMEN 2021 - ¡TODOS LOS MESES CAPTURADOS!
+-----+--------------------+
|month|     Ocupados_Brutos|
+-----+--------------------+
|    1|2.9562870281548284E7|
|    2|3.0824300907625232E7|
|    3| 3.086000140970879E7|
|    4| 3.022640163679197E7|
|    5| 3.048047239750262E7|
|    6| 3.047326513095185E7|
|    7|3.1213921084093824E7|
|    8| 3.201079222703463E7|
|    9|6.107936489289065...|
|   10| 3.283245746735253E7|
|   11|3.2293006439036146E7|
|   12|3.1911102909321398E7|
+-----+--------------------+



## SILVER 2022- RECONSTRUIDO

In [7]:
## ============================================================================
## 🛠️ SILVER 2022: RECONSTRUCCIÓN DEFINITIVA (RE-FIXED)
## ============================================================================
from pyspark.sql import functions as F
import re  # Usamos el motor de regex de Python para los nombres de archivos

year = 2022
bronze_path = f"Files/raw/dane/year={year}/"
silver_table = f"dane_silver_lh.labor_{year}"

print(f"🕵️‍♂️ Iniciando escaneo de {year}...")

# 1. Lectura total
df_raw = spark.read.format("text").option("recursiveFileLookup", "true").load(bronze_path) \
              .withColumn("file_name", F.input_file_name()) \
              .withColumn("fn_low", F.lower(F.col("file_name")))

# 2. Clasificación "Red de Pesca"
df_meta = df_raw.withColumn(
    "status_file",
    F.when(F.col("fn_low").rlike("deso|no%20ocu|no_ocu"), "desocupado")
    .when(F.col("fn_low").rlike("ocupa") & ~F.col("fn_low").contains("deso"), "ocupado")
    .otherwise("otro")
).filter(F.col("status_file").isin("ocupado", "desocupado")) \
 .filter(~F.col("fn_low").rlike("caracteristicas|ingresos|vivienda|fuerza|seguridad|formas|inactivos|juvenil|migracion"))

# 3. Procesamiento archivo por archivo
unique_files = [row.file_name for row in df_meta.select("file_name").distinct().collect()]

all_dfs = []
print(f"📂 Procesando {len(unique_files)} archivos de 2022...")

for fn in unique_files:
    # --- 🛠️ FIX AQUÍ: Usamos Python puro para el mes ---
    match = re.search(r"month=(\d+)", fn)
    if not match: continue
    m = int(match.group(1))

    df_file = df_meta.filter(F.col("file_name") == fn)
    label = df_file.select("status_file").limit(1).collect()[0][0]

    # Buscar el header
    header_row = df_file.filter(F.upper(F.col("value")).contains("DIRECTORIO")).limit(1).collect()
    if not header_row: continue
    
    val = header_row[0]['value']
    delim = ";" if ";" in val else ","
    cols = [c.strip().upper() for c in val.split(delim)]
    
    try:
        idx_peso = next(i for i, c in enumerate(cols) if "FEX" in c or "PESO" in c)
        idx_dsi = next((i for i, c in enumerate(cols) if "DSI" in c), None)
        
        df_proc = df_file.filter(~F.upper(F.col("value")).contains("DIRECTORIO")) \
            .withColumn("split_data", F.split(F.col("value"), delim)) \
            .withColumn("geo_source", F.when(F.col("fn_low").rlike("cabe|urban"), "cabecera")
                                     .when(F.col("fn_low").rlike("resto|rural"), "resto")
                                     .otherwise("area"))
        
        # Filtro DSI para desocupados
        if label == "desocupado" and idx_dsi is not None:
            df_proc = df_proc.filter(F.regexp_replace(F.trim(F.col("split_data")[idx_dsi]), '"', '') == "1")

        df_final = df_proc.select(
            F.lit(year).alias("year"),
            F.lit(m).alias("month"),
            F.lit(label).alias("status"),
            F.col("geo_source"),
            F.regexp_replace(F.regexp_replace(F.col("split_data")[idx_peso], '"', ''), ",", ".").cast("double").alias("weight")
        ).filter(F.col("weight").isNotNull())
        
        all_dfs.append(df_final)
    except Exception as e:
        # Si un archivo falla, lo saltamos pero avisamos
        print(f"   ⚠️ Error en archivo {fn.split('/')[-1]}: {str(e)[:50]}")
        continue

# 4. Unión y Guardado
if all_dfs:
    df_silver_2022 = all_dfs[0]
    for d in all_dfs[1:]: df_silver_2022 = df_silver_2022.unionByName(d)
    
    df_silver_2022 = df_silver_2022.withColumn("is_national_total", F.col("geo_source").isin("cabecera", "resto"))
    
    spark.sql(f"DROP TABLE IF EXISTS {silver_table}")
    df_silver_2022.write.format("delta").mode("overwrite").saveAsTable(silver_table)
    
    print(f"✅ SILVER {year} RECONSTRUIDO")
    df_silver_2022.groupBy("month", "status").count().orderBy("month", "status").show(24)
else:
    print("❌ No se pudo procesar ningún archivo. Revisa los delimitadores.")

StatementMeta(, ca0ff25c-c42f-481a-8f49-50adb8c09a33, 9, Finished, Available, Finished, False)

🕵️‍♂️ Iniciando escaneo de 2022...
📂 Procesando 24 archivos de 2022...
✅ SILVER 2022 RECONSTRUIDO
+-----+----------+-----+
|month|    status|count|
+-----+----------+-----+
|    1|desocupado| 5734|
|    1|   ocupado|31819|
|    2|desocupado| 5232|
|    2|   ocupado|32979|
|    3|desocupado| 4785|
|    3|   ocupado|32875|
|    4|desocupado| 4590|
|    4|   ocupado|31914|
|    5|desocupado| 4480|
|    5|   ocupado|32538|
|    6|desocupado| 4528|
|    6|   ocupado|32522|
|    7|desocupado| 4485|
|    7|   ocupado|31531|
|    8|desocupado| 4462|
|    8|   ocupado|31820|
|    9|desocupado| 4279|
|    9|   ocupado|31867|
|   10|desocupado| 3945|
|   10|   ocupado|30958|
|   11|desocupado| 3932|
|   11|   ocupado|31133|
|   12|desocupado| 4113|
|   12|   ocupado|30505|
+-----+----------+-----+



## SILVER 2023- RECONSTRUCCION

In [8]:
## ============================================================================
## 🚀 SILVER 2023: EL CIERRE DE LA SERIE HISTÓRICA
## ============================================================================
from pyspark.sql import functions as F
import re

year = 2023
bronze_path = f"Files/raw/dane/year={year}/"
silver_table = f"dane_silver_lh.labor_{year}"

print(f"🕵️‍♂️ Iniciando escaneo final de {year}...")

# 1. Lectura total
df_raw = spark.read.format("text").option("recursiveFileLookup", "true").load(bronze_path) \
              .withColumn("file_name", F.input_file_name()) \
              .withColumn("fn_low", F.lower(F.col("file_name")))

# 2. Clasificación
df_meta = df_raw.withColumn(
    "status_file",
    F.when(F.col("fn_low").rlike("deso|no%20ocu|no_ocu"), "desocupado")
    .when(F.col("fn_low").rlike("ocupa") & ~F.col("fn_low").contains("deso"), "ocupado")
    .otherwise("otro")
).filter(F.col("status_file").isin("ocupado", "desocupado")) \
 .filter(~F.col("fn_low").rlike("caracteristicas|ingresos|vivienda|fuerza|seguridad|formas|inactivos|juvenil"))

# 3. Procesamiento por archivo
unique_files = [row.file_name for row in df_meta.select("file_name").distinct().collect()]
all_dfs = []

for fn in unique_files:
    match = re.search(r"month=(\d+)", fn)
    if not match: continue
    m = int(match.group(1))

    df_file = df_meta.filter(F.col("file_name") == fn)
    label = df_file.select("status_file").limit(1).collect()[0][0]

    header_row = df_file.filter(F.upper(F.col("value")).contains("DIRECTORIO")).limit(1).collect()
    if not header_row: continue
    
    val = header_row[0]['value']
    delim = ";" if ";" in val else ","
    cols = [c.strip().upper() for c in val.split(delim)]
    
    try:
        idx_peso = next(i for i, c in enumerate(cols) if "FEX" in c or "PESO" in c)
        idx_dsi = next((i for i, c in enumerate(cols) if "DSI" in c), None)
        
        df_proc = df_file.filter(~F.upper(F.col("value")).contains("DIRECTORIO")) \
            .withColumn("split_data", F.split(F.col("value"), delim)) \
            .withColumn("geo_source", F.when(F.col("fn_low").rlike("cabe|urban"), "cabecera")
                                     .when(F.col("fn_low").rlike("resto|rural"), "resto")
                                     .otherwise("area"))
        
        # Filtro DSI para asegurar que no contamos inactivos en desocupados
        if label == "desocupado" and idx_dsi is not None:
            df_proc = df_proc.filter(F.regexp_replace(F.trim(F.col("split_data")[idx_dsi]), '"', '') == "1")

        df_final = df_proc.select(
            F.lit(year).alias("year"),
            F.lit(m).alias("month"),
            F.lit(label).alias("status"),
            F.col("geo_source"),
            F.regexp_replace(F.regexp_replace(F.col("split_data")[idx_peso], '"', ''), ",", ".").cast("double").alias("weight")
        ).filter(F.col("weight").isNotNull())
        
        all_dfs.append(df_final)
    except: continue

# 4. Unión y Guardado
if all_dfs:
    df_silver_2023 = all_dfs[0]
    for d in all_dfs[1:]: df_silver_2023 = df_silver_2023.unionByName(d)
    
    df_silver_2023 = df_silver_2023.withColumn("is_national_total", F.col("geo_source").isin("cabecera", "resto"))
    
    spark.sql(f"DROP TABLE IF EXISTS {silver_table}")
    df_silver_2023.write.format("delta").mode("overwrite").saveAsTable(silver_table)
    
    print(f"✅ SILVER {year} COMPLETADO")
    df_silver_2023.groupBy("month", "status").count().orderBy("month", "status").show(24)
else:
    print("❌ 2023 falló.")

StatementMeta(, ca0ff25c-c42f-481a-8f49-50adb8c09a33, 10, Finished, Available, Finished, False)

🕵️‍♂️ Iniciando escaneo final de 2023...
✅ SILVER 2023 COMPLETADO
+-----+----------+-----+
|month|    status|count|
+-----+----------+-----+
|    1|desocupado| 5114|
|    1|   ocupado|29695|
|    2|desocupado| 4591|
|    2|   ocupado|30726|
|    3|desocupado| 4247|
|    3|   ocupado|31009|
|    4|desocupado| 4207|
|    4|   ocupado|30654|
|    5|desocupado| 4040|
|    5|   ocupado|30517|
|    6|desocupado| 3880|
|    6|   ocupado|30535|
|    7|desocupado| 3822|
|    7|   ocupado|31018|
|    8|desocupado| 3695|
|    8|   ocupado|31086|
|    9|desocupado| 3613|
|    9|   ocupado|30586|
|   10|desocupado| 3467|
|   10|   ocupado|29797|
|   11|desocupado| 3606|
|   11|   ocupado|30479|
|   12|desocupado| 3683|
|   12|   ocupado|29717|
+-----+----------+-----+



## SILVER 2024- RECONSTRUIDO

In [3]:
## ============================================================================
## 🚀 SILVER 2024: EL GRAN RESCATE (ANTI-CODIFICACIÓN RARA)
## ============================================================================
from pyspark.sql import functions as F
import re

year = 2024
bronze_path = f"abfss://f5879c16-6832-4003-90dd-eb2c10497cc0@onelake.dfs.fabric.microsoft.com/ecf22981-1012-4b17-861f-62880aa61238/Files/raw/dane/year={year}/"
silver_table = f"dane_silver_lh.labor_{year}"

print(f"🕵️‍♂️ Iniciando reconstrucción total de {year}...")

# 1. Lectura masiva
df_raw = spark.read.format("text").option("recursiveFileLookup", "true").load(bronze_path) \
              .withColumn("file_name", F.input_file_name()) \
              .withColumn("fn_low", F.lower(F.col("file_name")))

# 2. Clasificación mejorada (2024 usa mucho 'No ocupados')
df_meta = df_raw.withColumn(
    "status_file",
    F.when(F.col("fn_low").rlike("deso|no%20ocu|no_ocu"), "desocupado")
    .when(F.col("fn_low").rlike("ocupa") & ~F.col("fn_low").contains("deso"), "ocupado")
    .otherwise("otro")
).filter(F.col("status_file").isin("ocupado", "desocupado")) \
 .filter(~F.col("fn_low").rlike("caracteristicas|generales|seguridad|salud|educaci|otras|formas|vivienda"))

# 3. Procesamiento archivo por archivo con detección de cabecera flexible
unique_files = [row.file_name for row in df_meta.select("file_name").distinct().collect()]
all_dfs = []

print(f"📂 Procesando {len(unique_files)} archivos clave encontrados...")

for fn in unique_files:
    # Extraer mes
    match = re.search(r"month=(\d+)", fn)
    if not match: continue
    m = int(match.group(1))

    df_file = df_meta.filter(F.col("file_name") == fn)
    label = df_file.select("status_file").limit(1).collect()[0][0]

    # Tomamos la primera fila para detectar el esquema (sea header o no)
    sample_rows = df_file.limit(2).collect()
    if not sample_rows: continue
    
    # Buscamos cuál de las primeras dos filas es el header (la que tenga FEX o DIRECTORIO)
    header_val = sample_rows[0]['value']
    if "FEX" not in header_val.upper() and "DIRECTORIO" not in header_val.upper() and len(sample_rows) > 1:
        header_val = sample_rows[1]['value']

    delim = ";" if ";" in header_val else ","
    cols = [c.strip().upper() for c in header_val.split(delim)]
    
    try:
        # Buscamos peso con un radar más amplio
        idx_peso = next(i for i, c in enumerate(cols) if any(x in c for x in ["FEX", "PESO", "FACTOR"]))
        idx_dsi = next((i for i, c in enumerate(cols) if "DSI" in c), None)
        
        # Filtramos la fila del header si aparece en los datos
        df_proc = df_file.filter(~F.col("value").contains(header_val[:20])) \
            .withColumn("split_data", F.split(F.col("value"), delim)) \
            .withColumn("geo_source", F.when(F.col("fn_low").rlike("cabe|urban"), "cabecera")
                                     .when(F.col("fn_low").rlike("resto|rural"), "resto")
                                     .otherwise("area"))
        
        if label == "desocupado" and idx_dsi is not None:
            # En 2024 el DSI puede venir con decimales .00, limpiamos todo
            df_proc = df_proc.filter(F.split(F.regexp_replace(F.trim(F.col("split_data")[idx_dsi]), '"', ''), "\.")[0] == "1")

        df_final = df_proc.select(
            F.lit(year).alias("year"),
            F.lit(m).alias("month"),
            F.lit(label).alias("status"),
            F.col("geo_source"),
            F.regexp_replace(F.regexp_replace(F.col("split_data")[idx_peso], '"', ''), ",", ".").cast("double").alias("weight")
        ).filter(F.col("weight").isNotNull())
        
        all_dfs.append(df_final)
    except Exception as e:
        continue

# 4. Unión y Guardado
if all_dfs:
    df_silver_2024 = all_dfs[0]
    for d in all_dfs[1:]: df_silver_2024 = df_silver_2024.unionByName(d)
    
    df_silver_2024 = df_silver_2024.withColumn("is_national_total", F.col("geo_source").isin("cabecera", "resto"))
    
    spark.sql(f"DROP TABLE IF EXISTS {silver_table}")
    df_silver_2024.write.format("delta").mode("overwrite").saveAsTable(silver_table)
    
    print(f"✅ SILVER {year} RECONSTRUIDO")
    df_silver_2024.groupBy("month", "status").count().orderBy("month", "status").show(24)
else:
    print("❌ 2024 sigue rebelde. Revisa si los archivos están vacíos.")

StatementMeta(, 8e992858-b40a-4b01-bc4c-b5b05e2ca10c, 5, Finished, Available, Finished, False)

🕵️‍♂️ Iniciando reconstrucción total de 2024...
📂 Procesando 24 archivos clave encontrados...
✅ SILVER 2024 RECONSTRUIDO
+-----+----------+-----+
|month|    status|count|
+-----+----------+-----+
|    1|desocupado| 4942|
|    1|   ocupado|28815|
|    2|desocupado| 4711|
|    2|   ocupado|29943|
|    3|desocupado| 4363|
|    3|   ocupado|29638|
|    4|desocupado| 4213|
|    4|   ocupado|29741|
|    5|desocupado| 4085|
|    5|   ocupado|30142|
|    6|desocupado| 4018|
|    6|   ocupado|29925|
|    7|desocupado| 3843|
|    7|   ocupado|29929|
|    8|desocupado| 3797|
|    8|   ocupado|29951|
|    9|desocupado| 3512|
|    9|   ocupado|29383|
|   10|desocupado| 3432|
|   10|   ocupado|29127|
|   11|desocupado| 3139|
|   11|   ocupado|28924|
|   12|desocupado| 3340|
|   12|   ocupado|28154|
+-----+----------+-----+



## SILVER 2025 Y 2026 JUNTOS: RESTAURADOS

In [4]:
## ============================================================================
## 🚀 SILVER 2025-2026: EL CIERRE DE LA SERIE
## ============================================================================
from pyspark.sql import functions as F
import re

new_years = [2025, 2026]

for y in new_years:
    print(f"\n🚀 Procesando Silver {y}...")
    bronze_path = f"abfss://f5879c16-6832-4003-90dd-eb2c10497cc0@onelake.dfs.fabric.microsoft.com/ecf22981-1012-4b17-861f-62880aa61238/Files/raw/dane/year={y}/"
    silver_table = f"dane_silver_lh.labor_{y}"

    try:
        df_raw = spark.read.format("text").option("recursiveFileLookup", "true").load(bronze_path) \
                      .withColumn("file_name", F.input_file_name()) \
                      .withColumn("fn_low", F.lower(F.col("file_name")))

        df_meta = df_raw.withColumn(
            "status_file",
            F.when(F.col("fn_low").rlike("deso|no%20ocu|no_ocu"), "desocupado")
            .when(F.col("fn_low").rlike("ocupa") & ~F.col("fn_low").contains("deso"), "ocupado")
            .otherwise("otro")
        ).filter(F.col("status_file").isin("ocupado", "desocupado")) \
         .filter(~F.col("fn_low").rlike("caracteristicas|generales|seguridad|salud|educaci|otras|formas|vivienda"))

        unique_files = [row.file_name for row in df_meta.select("file_name").distinct().collect()]
        all_dfs = []

        for fn in unique_files:
            match = re.search(r"month=(\d+)", fn)
            if not match: continue
            m = int(match.group(1))

            df_file = df_meta.filter(F.col("file_name") == fn)
            label = df_file.select("status_file").limit(1).collect()[0][0]

            sample_rows = df_file.limit(2).collect()
            if not sample_rows: continue
            
            header_val = sample_rows[0]['value']
            if "FEX" not in header_val.upper() and "DIRECTORIO" not in header_val.upper() and len(sample_rows) > 1:
                header_val = sample_rows[1]['value']

            delim = ";" if ";" in header_val else ","
            cols = [c.strip().upper() for c in header_val.split(delim)]
            
            idx_peso = next(i for i, c in enumerate(cols) if any(x in c for x in ["FEX", "PESO", "FACTOR"]))
            idx_dsi = next((i for i, c in enumerate(cols) if "DSI" in c), None)
            
            df_proc = df_file.filter(~F.col("value").contains(header_val[:20])) \
                .withColumn("split_data", F.split(F.col("value"), delim)) \
                .withColumn("geo_source", F.when(F.col("fn_low").rlike("cabe|urban"), "cabecera")
                                         .when(F.col("fn_low").rlike("resto|rural"), "resto")
                                         .otherwise("area"))
            
            if label == "desocupado" and idx_dsi is not None:
                df_proc = df_proc.filter(F.split(F.regexp_replace(F.trim(F.col("split_data")[idx_dsi]), '"', ''), "\.")[0] == "1")

            df_final = df_proc.select(
                F.lit(y).alias("year"),
                F.lit(m).alias("month"),
                F.lit(label).alias("status"),
                F.col("geo_source"),
                F.regexp_replace(F.regexp_replace(F.col("split_data")[idx_peso], '"', ''), ",", ".").cast("double").alias("weight")
            ).filter(F.col("weight").isNotNull())
            
            all_dfs.append(df_final)

        if all_dfs:
            df_silver = all_dfs[0]
            for d in all_dfs[1:]: df_silver = df_silver.unionByName(d)
            df_silver = df_silver.withColumn("is_national_total", F.col("geo_source").isin("cabecera", "resto"))
            
            spark.sql(f"DROP TABLE IF EXISTS {silver_table}")
            df_silver.write.format("delta").mode("overwrite").saveAsTable(silver_table)
            print(f"   ✅ Silver {y} guardado exitosamente.")
        else:
            print(f"   ⚠️ No se encontraron datos válidos para {y}.")
    except Exception as e:
        print(f"   ❌ Error en {y}: {str(e)[:100]}")

StatementMeta(, 8e992858-b40a-4b01-bc4c-b5b05e2ca10c, 6, Finished, Available, Finished, False)


🚀 Procesando Silver 2025...
   ✅ Silver 2025 guardado exitosamente.

🚀 Procesando Silver 2026...
   ✅ Silver 2026 guardado exitosamente.
